# 23 — Lightweight Back-Trajectory Reconstruction

April 2025 replacement.

In [ ]:

SITE_ID = "NHV"
SITE_NAME = "Newhaven ERF"
LAT = 50.80208
LON = 0.04961
DATE_FROM = "2025-04-02"
DATE_TO   = "2025-04-07"
TARGET_TIME_UTC = None
HOURS_BACK = 24
STEP_HOURS = 1
SCENE_CATALOG_CSV = "outputs/s5p/NHV_no2_offl_scene_catalog.csv"
OUTPUT_DIR = "outputs/back_trajectory"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": start_date, "end_date": end_date, "hourly": hourly_vars, "timezone": "UTC"}
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(OUTPUT_DIR) / "weather_cache"; CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

def fetch_weather(lat, lon):
    j=fetch_weather_cached(CACHE_DIR,lat,lon,DATE_FROM,DATE_TO,"wind_speed_10m,wind_direction_10m"); h=j.get("hourly",{})
    return pd.DataFrame({"time_utc":pd.to_datetime(h.get("time",[]),utc=True,errors="coerce"),"wind_speed_ms":h.get("wind_speed_10m",[]),"wind_dir_deg":h.get("wind_direction_10m",[])})
wind=fetch_weather(LAT,LON)
scene_df,_=safe_read_csv(SCENE_CATALOG_CSV)
if not scene_df.empty and "start_utc" in scene_df.columns:
    scene_df["start_utc"]=pd.to_datetime(scene_df["start_utc"],utc=True,errors="coerce"); scene_df=scene_df.dropna(subset=["start_utc"]).sort_values("start_utc").reset_index(drop=True)
target_ts=scene_df.iloc[0]["start_utc"] if (TARGET_TIME_UTC is None and not scene_df.empty) else (pd.Timestamp(TARGET_TIME_UTC,tz="UTC") if TARGET_TIME_UTC else pd.Timestamp(f"{DATE_FROM}T12:00:00Z"))


In [ ]:

KM_PER_DEG_LAT=111.32
def km_per_deg_lon(lat_deg): return 111.32*math.cos(math.radians(lat_deg))
def nearest_hour(ts,wind_df): idx=(wind_df["time_utc"]-ts).abs().idxmin(); return wind_df.loc[idx]
def step_back(lat,lon,wind_speed_ms,wind_dir_deg,hours=1):
    distance_km=(wind_speed_ms*hours*3600.0)/1000.0; bearing_deg=wind_dir_deg
    dlat=(distance_km*math.cos(math.radians(bearing_deg)))/KM_PER_DEG_LAT
    dlon=(distance_km*math.sin(math.radians(bearing_deg)))/km_per_deg_lon(lat)
    return lat+dlat,lon+dlon,distance_km
rows=[]; cur_lat,cur_lon,cur_time=LAT,LON,target_ts
rows.append({"step":0,"time_utc":cur_time,"lat":cur_lat,"lon":cur_lon,"wind_speed_ms":np.nan,"wind_dir_deg":np.nan,"distance_step_km":0.0})
for i in range(1,HOURS_BACK+1,STEP_HOURS):
    w=nearest_hour(cur_time,wind); ws=float(w["wind_speed_ms"]) if pd.notna(w["wind_speed_ms"]) else 0.0; wd=float(w["wind_dir_deg"]) if pd.notna(w["wind_dir_deg"]) else 0.0
    next_lat,next_lon,dkm=step_back(cur_lat,cur_lon,ws,wd,hours=STEP_HOURS); cur_time=cur_time-pd.Timedelta(hours=STEP_HOURS); cur_lat,cur_lon=next_lat,next_lon
    rows.append({"step":i,"time_utc":cur_time,"lat":cur_lat,"lon":cur_lon,"wind_speed_ms":ws,"wind_dir_deg":wd,"distance_step_km":dkm})
traj=pd.DataFrame(rows); traj.to_csv(Path(OUTPUT_DIR)/f"{SITE_ID}_back_trajectory.csv",index=False); display(traj.head(10))


In [ ]:

features=[{"type":"Feature","geometry":{"type":"Point","coordinates":[float(r["lon"]),float(r["lat"])]},"properties":{"step":int(r["step"]),"time_utc":str(r["time_utc"])}} for _,r in traj.iterrows()]
line={"type":"Feature","geometry":{"type":"LineString","coordinates":[[float(r["lon"]),float(r["lat"])] for _,r in traj.iterrows()]},"properties":{"site_id":SITE_ID,"site_name":SITE_NAME,"target_time_utc":str(target_ts)}}
(Path(OUTPUT_DIR)/f"{SITE_ID}_back_trajectory.geojson").write_text(json.dumps({"type":"FeatureCollection","features":[line]+features},indent=2),encoding="utf-8")
fig,ax=plt.subplots(figsize=(6,6)); ax.plot(traj["lon"],traj["lat"],marker="o"); ax.scatter([LON],[LAT],marker="x",s=120); ax.set_title(f"{SITE_NAME} lightweight back trajectory"); ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/f"{SITE_ID}_back_trajectory.png"; fig.savefig(plot_path,dpi=150); plt.show()
(Path(OUTPUT_DIR)/f"{SITE_ID}_back_trajectory_manifest.json").write_text(json.dumps({"rows":int(len(traj)),"target_time_utc":str(target_ts)},indent=2),encoding="utf-8")
print("Saved:",plot_path)
